In [ ]:
# Packages
import pyomo.environ as pyo
from pyomo.environ import quicksum
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import time
import copy
import os
import pickle
from collections import defaultdict
from datetime import datetime
from pathlib import Path

In [ ]:
# Model Controllers (False: Deactivated, True: Activated)
Price_Tech_Clustering = False 
Elec_Gas_Coupling = False # Should be False if Price_Tech_Clustering is False
MGA = False # Should be False if Price_Tech_Clustering is False
Out_of_Smaple = False # Should be False if MGA is False

In [ ]:
# Load Data
start_time = time.time()
if MGA and Out_of_Smaple:
    with open('Input Data/Multi (2025)/G_Offers.pkl', 'rb') as f1, \
        open('Input Data/Multi (2025)/D_Bids.pkl', 'rb') as f2, \
        open('Input Data/Multi (2025)/G_Offers3.pkl', 'rb') as f3:
        G_Offers = pickle.load(f1)
        D_Bids = pickle.load(f2)
        G_Offers2 = pickle.load(f3)
else:
    with open('Input Data/Multi (Year)/G_Offers.pkl', 'rb') as f1, \
        open('Input Data/Multi (Year)/D_Bids.pkl', 'rb') as f2, \
        open('Input Data/Multi (Year)/G_Offers3.pkl', 'rb') as f3:
        G_Offers = pickle.load(f1)
        D_Bids = pickle.load(f2)
        G_Offers2 = pickle.load(f3)

    # Add Missing Samples
    if not MGA:
        date = "31/03/2024"
        D_Bids[date][24] = copy.deepcopy(D_Bids[date][23])
        G_Offers[date][24] = copy.deepcopy(G_Offers[date][23])
        G_Offers2[date][24] = copy.deepcopy(G_Offers2[date][23])

In [ ]:
# Find Min and Max MCP Prices
start_time = time.time()
min_price_ES = float('inf')
max_price_ES = float('-inf')
min_price_PT = float('inf')
max_price_PT = float('-inf')
for data in D_Bids.values():
    for D_hour in data.values():
        es = D_hour.get('Spain')
        if es:
            arr = np.concatenate([np.asarray(entry['market_clearing_price']) for entry in es.values()])
            min_price_ES = min(min_price_ES, arr.min())
            max_price_ES = max(max_price_ES, arr.max())
        pt = D_hour.get('Portugal')
        if pt:
            arr = np.concatenate([np.asarray(entry['market_clearing_price']) for entry in pt.values()])
            min_price_PT = min(min_price_PT, arr.min())
            max_price_PT = max(max_price_PT, arr.max())
print("\nExecution time:", time.time() - start_time, "seconds")

In [ ]:
# Filtering Input Data
start_time = time.time()
N_T=24
if MGA and Out_of_Smaple:
    chosen_month_years = [
        "01/2025", 
        "02/2025" 
    ]
    month_days = {
        "01/2025": 31,
        "02/2025": 28
    }
elif MGA and not Out_of_Smaple:
    chosen_month_years = [
        "01/2024",
        "02/2024"
    ]
    month_days = {
        "01/2024": 31,
        "02/2024": 28
    }
else:
    chosen_month_years = [
        "01/2024", 
        "02/2024", 
        "03/2024", 
        "04/2024", 
        "05/2024", 
        "06/2024",
        "07/2024", 
        "08/2024", 
        "09/2024", 
        "10/2024", 
        "11/2024", 
        "12/2024"
    ]
    month_days = {
        "01/2024": 31,
        "02/2024": 29, 
        "03/2024": 31,
        "04/2024": 30,
        "05/2024": 31,
        "06/2024": 30,
        "07/2024": 31,
        "08/2024": 31,
        "09/2024": 30,
        "10/2024": 31,
        "11/2024": 30,
        "12/2024": 31
}
def build_selected_dates(chosen_month_years, month_days):
    return {
        f"{d:02d}/{my[0:2]}/{my[3:]}"
        for my in chosen_month_years
        for d in range(1, int(month_days.get(my, 0)) + 1)
    }
SELECTED_DATES = build_selected_dates(chosen_month_years, month_days)
def filter_date_hour(data, selected_dates, n_t):
    return {
        date: {
            t: payload for t, payload in hours_dict.items()
            if (lambda ti: ti is not None and 1 <= ti <= n_t)(
                int(t) if isinstance(t, (str, int)) and str(t).isdigit() else None
            )
        }
        for date, hours_dict in data.items()
        if date in selected_dates and any(
            (lambda ti: ti is not None and 1 <= ti <= n_t)(
                int(t) if isinstance(t, (str, int)) and str(t).isdigit() else None
            ) for t in hours_dict.keys()
        )
    }
G_Offers  = filter_date_hour(G_Offers,  SELECTED_DATES, N_T)
D_Bids    = filter_date_hour(D_Bids,    SELECTED_DATES, N_T)
G_Offers2 = filter_date_hour(G_Offers2, SELECTED_DATES, N_T)
print("Kept dates:", sorted(G_Offers.keys()))
print("Example hours kept:", sorted(next(iter(G_Offers.values())).keys()) if G_Offers else "none")
print("\nExecution time:", time.time() - start_time, "seconds")

In [ ]:
# Load Gas Prices
start_time = time.time()
df = pd.read_excel(
    'Input Data/MIBGAS_Data_2024.xlsx',
    sheet_name='MIBGAS Indexes',
    usecols=[0, 1],
    header=None,
    names=['Date', 'Value'],
    skiprows=1
)
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
df = df.sort_values('Date', ignore_index=True)
_valuemin = df['Value'].min()
_valuemax = df['Value'].max()
_valuemean = df['Value'].mean()
gas_price = df
v = df['Value']
gas_price['Normalized_min'] = v / _valuemin
gas_price['Normalized_max'] = v / _valuemax
gas_price['Normalized_mean'] = v / _valuemean
selected_date_objs = set(
    pd.to_datetime(d, dayfirst=True, errors='coerce')
    for d in SELECTED_DATES
)

# Filter Gas Prices
df = df[df['Date'].isin(selected_date_objs)].reset_index(drop=True)
gas_price = gas_price[gas_price['Date'].isin(selected_date_objs)].reset_index(drop=True)
print("\nExecution time:", time.time() - start_time, "seconds")

In [ ]:
# Algorithm 1 (Based on Market Clearing Prices)
start_time = time.time()
N_B = 20
observed_mcp_ES = []
observed_mcp_PT = []
all_border_exp = []
def parse_calendar_date(date_str):
    if isinstance(date_str, datetime):
        return date_str
    try:
        return datetime.strptime(date_str, "%d/%m/%Y")
    except Exception:
        try:
            return datetime.strptime(date_str, "%Y-%m-%d")
        except Exception:
            return pd.to_datetime(date_str, dayfirst=True, errors='coerce')
sorted_dates = sorted(D_Bids.keys(), key=parse_calendar_date)
for selected_date in sorted_dates:
    D_sel = D_Bids[selected_date]
    for t in D_sel:
        D_hour = D_sel[t]
        es = D_hour.get('Spain')
        pt = D_hour.get('Portugal')
        if es:
            for entry in es.values():
                arr = np.asarray(entry['market_clearing_price'])
                observed_mcp_ES.extend(arr)
                all_border_exp.extend(np.abs(np.asarray(entry['border_exchange'])))
        if pt:
            for entry in pt.values():
                arr = np.asarray(entry['market_clearing_price'])
                observed_mcp_PT.extend(arr)

# Spain Clustering
if observed_mcp_ES:
    min_price = min_price_ES
    max_price = max_price_ES
    print('Zone: Spain')
    print(f"Minimum price value: {min_price}")
    print(f"Maximum price value: {max_price}")
slots_ES = None
slot_indices_by_datetime_ES = {}
reverse_slot_idx_map_ES = {}
slot_mode = 'equal'
if observed_mcp_ES:
    if slot_mode == 'equal':
        slots_ES = np.linspace(min_price, max_price, N_B + 1)
    elif slot_mode == 'random':
        rng = np.random.default_rng(seed=42)
        slot_width = (max_price - min_price) / N_B
        deviations = rng.uniform(-1, 1, N_B - 1) * slot_width
        boundaries = np.cumsum(np.full(N_B-1, slot_width) + deviations)
        boundaries = np.clip(boundaries, 0, max_price - min_price)
        slots_ES = np.array([min_price] + list(min_price + boundaries) + [max_price])
        slots_ES[0], slots_ES[-1] = min_price, max_price
        for i in range(1, len(slots_ES)):
            if slots_ES[i] <= slots_ES[i-1]:
                slots_ES[i] = slots_ES[i-1] + 1e-8
    else:
        raise ValueError(f"Unknown slot_mode: {slot_mode}")
    for selected_date in D_Bids:
        d_sel = D_Bids[selected_date]
        for t in d_sel:
            es = d_sel[t].get('Spain')
            if es:
                for entry in es.values():
                    mcp_prices = np.asarray(entry['market_clearing_price'])
                    indices = np.digitize(mcp_prices, slots_ES, right=False) - 1
                    indices = np.clip(indices, 0, len(slots_ES) - 2)
                    dt_key = (selected_date, t)
                    for idx in np.unique(indices):
                        slot_indices_by_datetime_ES.setdefault(idx, []).append(dt_key)
    for idx in sorted(slot_indices_by_datetime_ES):
        slot_min, slot_max = slots_ES[idx], slots_ES[idx + 1] if idx + 1 < len(slots_ES) else slots_ES[-1]
        dts = set(slot_indices_by_datetime_ES[idx])
        if slot_max <= slot_min:
            print(f"Slot {idx+1}: [{slot_min:.2f}, {slot_max:.2f}) -> Date/Times: {dts} (INVALID: min >= max!)")
        else:
            print(f"Slot {idx+1}: [{slot_min:.2f}, {slot_max:.2f}) -> Date/Times: {dts} (OK)")
    for slot_idx, dt_hr_list in slot_indices_by_datetime_ES.items():
        for dt, hr in dt_hr_list:
            reverse_slot_idx_map_ES.setdefault(dt, {})[hr] = slot_idx + 1

# Portugal Clustering
if observed_mcp_PT:
    min_price = min_price_PT
    max_price = max_price_PT
    print('\nZone: Portugal')
    print(f"Minimum price value: {min_price}")
    print(f"Maximum price value: {max_price}")
slots_PT = None
slot_indices_by_datetime_PT = {}
reverse_slot_idx_map_PT = {}
if observed_mcp_PT:
    if slot_mode == 'equal':
        slots_PT = np.linspace(min_price, max_price, N_B + 1)
    elif slot_mode == 'random':
        rng = np.random.default_rng(seed=42)
        slot_width = (max_price - min_price) / N_B
        deviations = rng.uniform(-1, 1, N_B - 1) * slot_width
        boundaries = np.cumsum(np.full(N_B-1, slot_width) + deviations)
        boundaries = np.clip(boundaries, 0, max_price - min_price)
        slots_PT = np.array([min_price] + list(min_price + boundaries) + [max_price])
        slots_PT[0], slots_PT[-1] = min_price, max_price
        for i in range(1, len(slots_PT)):
            if slots_PT[i] <= slots_PT[i-1]:
                slots_PT[i] = slots_PT[i-1] + 1e-8
    else:
        raise ValueError(f"Unknown slot_mode: {slot_mode}")
    for selected_date in D_Bids:
        d_sel = D_Bids[selected_date]
        for t in d_sel:
            pt = d_sel[t].get('Portugal')
            if pt:
                for entry in pt.values():
                    mc_prices = np.asarray(entry['market_clearing_price'])
                    indices = np.digitize(mc_prices, slots_PT, right=False) - 1
                    indices = np.clip(indices, 0, len(slots_PT) - 2)
                    dt_key = (selected_date, t)
                    for idx in np.unique(indices):
                        slot_indices_by_datetime_PT.setdefault(idx, []).append(dt_key)
    for idx in sorted(slot_indices_by_datetime_PT):
        slot_min, slot_max = slots_PT[idx], slots_PT[idx + 1] if idx + 1 < len(slots_PT) else slots_PT[-1]
        dts = set(slot_indices_by_datetime_PT[idx])
        if slot_max <= slot_min:
            print(f"Slot {idx+1}: [{slot_min:.2f}, {slot_max:.2f}) -> Date/Times: {dts} (INVALID: min >= max!)")
        else:
            print(f"Slot {idx+1}: [{slot_min:.2f}, {slot_max:.2f}) -> Date/Times: {dts} (OK)")
    for slot_idx, dt_hr_list in slot_indices_by_datetime_PT.items():
        for dt, hr in dt_hr_list:
            reverse_slot_idx_map_PT.setdefault(dt, {})[hr] = slot_idx + 1
G_Offers_clustered = {}
slots_dict = {'Spain': slots_ES if observed_mcp_ES else None, 'Portugal': slots_PT if observed_mcp_PT else None}
reverse_maps = {'Spain': reverse_slot_idx_map_ES, 'Portugal': reverse_slot_idx_map_PT}
for selected_date in G_Offers:
    G_Offers_clustered[selected_date] = {}
    for t in G_Offers[selected_date]:
        G_Offers_clustered[selected_date][t] = {}
        for country in G_Offers[selected_date][t]:
            slots = slots_dict[country]
            slot_idx = reverse_maps[country].get(selected_date, {}).get(t, None)
            G_Offers_clustered[selected_date][t][country] = {}
            for type in G_Offers[selected_date][t][country]:
                offers = G_Offers[selected_date][t][country][type]
                quant = np.asarray(offers['bid_quantity'])
                delivered = np.asarray(offers['activated_portion'])
                prices = np.asarray(offers['bid_price'])
                n_slots = len(slots) - 1 if slots is not None else 0
                slot_quantities = np.zeros(n_slots)
                slot_delivered = np.zeros(n_slots)
                slot_prices = np.zeros(n_slots)
                idx = (slot_idx-1) if slot_idx else None
                if idx is not None and 0 <= idx < n_slots:
                    slot_quantities[idx] = quant.sum()
                    slot_delivered[idx] = (quant * delivered).sum()/quant.sum() if quant.sum() != 0 else 0
                    slot_prices[idx] = prices.max() if prices.size else 0
                G_Offers_clustered[selected_date][t][country][type] = {
                    'bid_quantity': slot_quantities.tolist(),
                    'activated_portion': slot_delivered.tolist(),
                    'bid_price': slot_prices.tolist()
                }
G_Offers=G_Offers_clustered
print("\nExecution time:", time.time() - start_time, "seconds")

In [ ]:
# Algorithm 2 (Based on Market Clearing Prices + Technology)
if Price_Tech_Clustering:
    start_time = time.time()
    insertions_by_slot_ES = defaultdict(lambda: {'max_bids': 0, 'insertions': []})
    insertions_by_slot_PT = defaultdict(lambda: {'max_bids': 0, 'insertions': []})
    for selected_date, offers_t in G_Offers_clustered.items():
        for t, offers_c in offers_t.items():
            for country in offers_c.keys():
                if country == 'Spain':
                    slot_idx = reverse_slot_idx_map_ES.get(selected_date, {}).get(t, None)
                elif country == 'Portugal':
                    slot_idx = reverse_slot_idx_map_PT.get(selected_date, {}).get(t, None)
                else:
                    continue
                try:
                    g3_data = G_Offers2[selected_date][t][country]['Generation']
                    g3_quantities = g3_data['bid_quantity']
                    num_g3_bids = len(g3_quantities)
                except Exception:
                    g3_data = {'bid_quantity': [0.0]*N_B, 'bid_price': [0.0]*N_B, 'activated_portion': [0.0]*N_B}
                    g3_quantities = g3_data['bid_quantity']
                    num_g3_bids = len(g3_quantities)
                if num_g3_bids > 0:
                    target_dict = insertions_by_slot_ES if country == 'Spain' else insertions_by_slot_PT
                    if slot_idx not in target_dict or num_g3_bids > target_dict[slot_idx]['max_bids']:
                        target_dict[slot_idx]['max_bids'] = num_g3_bids
                    target_dict[slot_idx]['insertions'].append((selected_date, t, num_g3_bids, g3_data))
    def compute_insertions(insertions_by_slot):
        order = sorted(k for k in insertions_by_slot if k is not None)
        insertions_before_slot = {}
        cumulative = 0
        for slot_idx in order:
            insertions_before_slot[slot_idx] = cumulative
            cumulative += insertions_by_slot[slot_idx]['max_bids'] - 1
        return insertions_before_slot

    # Spain Zone
    if slots_dict['Spain'] is not None:
        insertions_before_slot_ES = compute_insertions(insertions_by_slot_ES)
        max_slots_ES = (len(slots_dict['Spain']) - 1) + sum(v['max_bids'] - 1 for v in insertions_by_slot_ES.values())
        total_insertions_ES = max_slots_ES - (len(slots_dict['Spain']) - 1)
        print('Zone: Spain')
        print(f"Found insertions at {len(insertions_by_slot_ES)} slot positions")
        print(f"Total insertions: {total_insertions_ES} slots added")
        print(f"Maximum slots needed: {max_slots_ES}")
        insertion_lookup_ES = {}
        for slot_idx, slot_data in insertions_by_slot_ES.items():
            for selected_date, t, num_bids, g3_data in slot_data['insertions']:
                insertion_lookup_ES[(selected_date, t)] = (slot_idx, slot_data['max_bids'], g3_data)

        # Thermal Blocks
        num_blocks1 = 6
        bid_block_numbers1_ES = set()
        for (selected_date, t), (insert_slot_idx, max_bids_for_slot, g3_data) in insertion_lookup_ES.items():
            if max_bids_for_slot >= num_blocks1-1:
                shifts_before = insertions_before_slot_ES.get(insert_slot_idx, 0)
                b_n = (insert_slot_idx - 1) + shifts_before + num_blocks1
                bid_block_numbers1_ES.add(b_n)

        # Target Blocks
        num_blocks2 = 6
        bid_block_numbers2_ES = set()
        for (selected_date, t), (insert_slot_idx, max_bids_for_slot, g3_data) in insertion_lookup_ES.items():
            if max_bids_for_slot >= num_blocks2-1:
                shifts_before = insertions_before_slot_ES.get(insert_slot_idx, 0)
                b_n = (insert_slot_idx - 1) + shifts_before + num_blocks2
                bid_block_numbers2_ES.add(b_n)
        bid_block_numbers1_ES = sorted(list(bid_block_numbers1_ES))
        bid_block_numbers2_ES = sorted(list(bid_block_numbers2_ES))

    # Portugal Zone
    if slots_dict['Portugal'] is not None:
        insertions_before_slot_PT = compute_insertions(insertions_by_slot_PT)
        max_slots_PT = (len(slots_dict['Portugal']) - 1) + sum(v['max_bids'] - 1 for v in insertions_by_slot_PT.values())
        total_insertions_PT = max_slots_PT - (len(slots_dict['Portugal']) - 1)
        print('\nZone: Portugal')
        print(f"Found insertions at {len(insertions_by_slot_PT)} slot positions")
        print(f"Total insertions: {total_insertions_PT} slots added")
        print(f"Maximum slots needed: {max_slots_PT}")
        insertion_lookup_PT = {}
        for slot_idx, slot_data in insertions_by_slot_PT.items():
            for selected_date, t, num_bids, g3_data in slot_data['insertions']:
                insertion_lookup_PT[(selected_date, t)] = (slot_idx, slot_data['max_bids'], g3_data)

        # Thermal Blocks
        num_blocks1 = 6
        bid_block_numbers1_PT = set()
        for (selected_date, t), (insert_slot_idx, max_bids_for_slot, g3_data) in insertion_lookup_PT.items():
            if max_bids_for_slot >= num_blocks1-1:
                shifts_before = insertions_before_slot_PT.get(insert_slot_idx, 0)
                b_n = (insert_slot_idx - 1) + shifts_before + num_blocks1
                bid_block_numbers1_PT.add(b_n)

        # Target Blocks
        num_blocks2 = 6
        bid_block_numbers2_PT = set()
        for (selected_date, t), (insert_slot_idx, max_bids_for_slot, g3_data) in insertion_lookup_PT.items():
            if max_bids_for_slot >= num_blocks2-1:
                shifts_before = insertions_before_slot_PT.get(insert_slot_idx, 0)
                b_n = (insert_slot_idx - 1) + shifts_before + num_blocks2
                bid_block_numbers2_PT.add(b_n)
        bid_block_numbers1_PT = sorted(list(bid_block_numbers1_PT))
        bid_block_numbers2_PT = sorted(list(bid_block_numbers2_PT))

    # Add New Bid Blocks
    G_Offers_clustered_expanded = {}
    for selected_date, offers_t in G_Offers_clustered.items():
        for t, offers_c in offers_t.items():
            for country, offers_country in offers_c.items():
                if country == 'Spain':
                    slot_idx = reverse_slot_idx_map_ES.get(selected_date, {}).get(t, None)
                    if slot_idx is None:
                        continue
                    max_slots = max_slots_ES
                    ins_before = insertions_before_slot_ES
                    insert_lookup = insertion_lookup_ES
                    ins_by_slot = insertions_by_slot_ES
                elif country == 'Portugal':
                    slot_idx = reverse_slot_idx_map_PT.get(selected_date, {}).get(t, None)
                    if slot_idx is None:
                        continue
                    max_slots = max_slots_PT
                    ins_before = insertions_before_slot_PT
                    insert_lookup = insertion_lookup_PT
                    ins_by_slot = insertions_by_slot_PT
                else:
                    continue
                if selected_date not in G_Offers_clustered_expanded:
                    G_Offers_clustered_expanded[selected_date] = {}
                if t not in G_Offers_clustered_expanded[selected_date]:
                    G_Offers_clustered_expanded[selected_date][t] = {}
                G_Offers_clustered_expanded[selected_date][t][country] = {}
                for type, clustered_data in offers_country.items():
                    orig_quantities = clustered_data['bid_quantity']
                    orig_delivered = clustered_data['activated_portion']
                    orig_prices = clustered_data['bid_price']
                    new_quantities = [0] * max_slots
                    new_delivered = [0] * max_slots
                    new_prices = [0] * max_slots
                    key_pair = (selected_date, t)
                    if key_pair in insert_lookup:
                        insert_slot_idx, max_bids_for_slot, g3_data = insert_lookup[key_pair]
                        shifts_before = ins_before.get(insert_slot_idx, 0)
                        insertion_pos = (insert_slot_idx - 1) + shifts_before
                        for orig_idx in range(insert_slot_idx - 1):
                            shifts_at_this_pos = sum(
                                ins_by_slot[s]['max_bids'] - 1
                                for s in ins_by_slot if s is not None and s <= (orig_idx + 1)
                            )
                            new_idx = orig_idx + shifts_at_this_pos
                            if orig_idx < len(orig_quantities) and new_idx < max_slots:
                                new_quantities[new_idx] = orig_quantities[orig_idx]
                                new_delivered[new_idx] = orig_delivered[orig_idx]
                                new_prices[new_idx] = orig_prices[orig_idx]
                        for i, (q, d, p) in enumerate(zip(g3_data['bid_quantity'], g3_data['activated_portion'], g3_data['bid_price'])):
                            idx = insertion_pos + i
                            if idx < max_slots:
                                new_quantities[idx] = q
                                new_delivered[idx] = d
                                new_prices[idx] = p
                        for orig_idx in range(insert_slot_idx, len(orig_quantities)):
                            shifts_at_this_pos = sum(
                                ins_by_slot[s]['max_bids'] - 1
                                for s in ins_by_slot if s is not None and s <= (orig_idx + 1)
                            )
                            new_idx = orig_idx + shifts_at_this_pos
                            if new_idx < max_slots:
                                new_quantities[new_idx] = orig_quantities[orig_idx]
                                new_delivered[new_idx] = orig_delivered[orig_idx]
                                new_prices[new_idx] = orig_prices[orig_idx]
                    else:
                        for orig_idx in range(len(orig_quantities)):
                            shifts_at_this_pos = sum(
                                ins_by_slot[s]['max_bids'] - 1
                                for s in ins_by_slot if s is not None and s <= (orig_idx + 1)
                            )
                            new_idx = orig_idx + shifts_at_this_pos
                            if new_idx < max_slots:
                                new_quantities[new_idx] = orig_quantities[orig_idx]
                                new_delivered[new_idx] = orig_delivered[orig_idx]
                                new_prices[new_idx] = orig_prices[orig_idx]
                    G_Offers_clustered_expanded[selected_date][t][country][type] = {
                        'bid_quantity': new_quantities,
                        'activated_portion': new_delivered,
                        'bid_price': new_prices
                    }
    G_Offers = G_Offers_clustered_expanded
    print("\nExecution time:", time.time() - start_time, "seconds")

In [ ]:
# Generate Final Bids/Offers
start_time = time.time()
Q_init_Spain = {}
Q_init_Portugal = {}
lambda_init_Spain = {}
lambda_init_Portugal = {}
X_init_Spain = {}
X_init_Portugal = {}
dd = 0
for chosen_month_year in chosen_month_years:
    N_D = month_days.get(chosen_month_year, 0)
    for d in range(1, N_D+1):
        dd += 1
        selected_date = f"{d:02d}/{chosen_month_year[0:2]}/{chosen_month_year[3:]}"
        for t in range(1, N_T+1):
            countries = G_Offers[selected_date][t]
            for country in countries:
                block_num = 0
                g_data = G_Offers[selected_date][t][country]
                if 'Generation' in g_data:
                    for quantity, price, delivered in zip(g_data['Generation']['bid_quantity'], g_data['Generation']['bid_price'], g_data['Generation']['activated_portion']):
                        block_num += 1
                        if country == 'Spain':
                            Q_init_Spain[block_num, dd, t] = -quantity
                            lambda_init_Spain[block_num, dd, t] = price
                            X_init_Spain[block_num, dd, t] = delivered
                        elif country == 'Portugal':
                            Q_init_Portugal[block_num, dd, t] = -quantity
                            lambda_init_Portugal[block_num, dd, t] = price
                            X_init_Portugal[block_num, dd, t] = delivered
                if t in D_Bids[selected_date] and country in D_Bids[selected_date][t]:
                    d_data = D_Bids[selected_date][t][country]
                    if 'Demand' in d_data:
                        for quantity, price, delivered in zip(d_data['Demand']['comulative_bid_quantities'], d_data['Demand']['market_clearing_price'], d_data['Demand']['comulative_activated_portions']):
                            block_num += 1
                            if country == 'Spain':
                                Q_init_Spain[block_num, dd, t] = quantity
                                lambda_init_Spain[block_num, dd, t] = price
                                X_init_Spain[block_num, dd, t] = delivered
                            elif country == 'Portugal':
                                Q_init_Portugal[block_num, dd, t] = quantity
                                lambda_init_Portugal[block_num, dd, t] = price
                                X_init_Portugal[block_num, dd, t] = delivered
N_Spain = max((b for (b, d, t) in Q_init_Spain), default=0)
N_Portugal = max((b for (b, d, t) in Q_init_Portugal), default=0)
dd = 0
Q_final = {}
lambda_final = {}
X_final = {}
for chosen_month_year in chosen_month_years:
    N_D = month_days.get(chosen_month_year, 0)
    for d in range(1, N_D + 1):
        dd += 1
        for t in range(1, N_T + 1):
            for b in range(1, N_Spain + N_Portugal + 1):
                if b <= N_Spain:
                    Q_final[b,dd,t] = Q_init_Spain.get((b,dd,t), 0)
                    lambda_final[b,dd,t] = lambda_init_Spain.get((b,dd,t), 0)
                    X_final[b,dd,t] = X_init_Spain.get((b,dd,t), 0) if Q_final[b,dd,t] != 0 else 0
                else:
                    portugal_b = b - N_Spain
                    Q_final[b,dd,t] = Q_init_Portugal.get((portugal_b,dd,t), 0)
                    lambda_final[b,dd,t] = lambda_init_Portugal.get((portugal_b,dd,t), 0)
                    X_final[b,dd,t] = X_init_Portugal.get((portugal_b,dd,t), 0) if Q_final[b,dd,t] != 0 else 0
zone_mapping = [('Spain',b) for b in range(1,N_Spain+1)] + \
            [('Portugal',b) for b in range(N_Spain+1,N_Spain+N_Portugal+1)]
print("\nExecution time:", time.time() - start_time, "seconds")

In [ ]:
# Load Results
start_time = time.time()
pickle_dir = 'IO Output'
results_list = []
def load_pickle_if_exists(filename):
    filepath = os.path.join(pickle_dir, filename)
    if os.path.isfile(filepath):
        with open(filepath, 'rb') as f:
            return pickle.load(f), filepath
    return None, None
if Elec_Gas_Coupling:
    res, filepath = load_pickle_if_exists('IO-Recovered Merit Orders (Under Elec-Gas Coupling).pkl')
    if res is not None:
        results_list.append(filepath)
else:
    res, filepath = load_pickle_if_exists('IO-Recovered Merit Orders (No Elec-Gas Coupling).pkl')
    if res is not None:
        results_list.append(filepath)
if MGA:
    n = 1
    while True:
        res, filepath = load_pickle_if_exists(f'Alternative {n}.pkl')
        if res is None:
            break
        results_list.append(filepath)
        n += 1
lambda_IO = {}
lambda_demand = {}
for file_idx, filename in enumerate(results_list):
    with open(filename, 'rb') as f:
        result = pickle.load(f)
    for z in ['Spain', 'Portugal']:
        if z == 'Spain':
            for b in range(1, int(N_Spain)):
                if file_idx == 0:
                    lambda_IO[(file_idx,z,b)] = result['lambda_IO']['lambda_IO'][z][b]
                else:
                    lambda_IO[(file_idx,z,b)] = result['lambda_MGA']['lambda_MGA'][z][b]
            for d in range(1, dd+1):
                for t in range(1, N_T+1):
                    lambda_demand[(file_idx,z,d,t)] = result['demand']['demand'][z][d][t]
        elif z == 'Portugal':
            for b in range(int(N_Spain)+1, 2*int(N_Spain)):
                if file_idx == 0:
                    lambda_IO[(file_idx,z,b)] = result['lambda_IO']['lambda_IO'][z][b]
                else:
                    lambda_IO[(file_idx,z,b)] = result['lambda_MGA']['lambda_MGA'][z][b]
            for d in range(1, dd+1):
                for t in range(1, N_T+1):
                    lambda_demand[(file_idx,z,d,t)] = result['demand']['demand'][z][d][t]
    print(filename)
print("\nExecution time:", time.time() - start_time, "seconds")

In [ ]:
# Create Forward Model
start_time = time.time()
model = pyo.ConcreteModel()
model.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)

# Sets
model.B = pyo.RangeSet(1,max(b for b,d,t in Q_final.keys()))
model.T = pyo.RangeSet(1, N_T)
model.Z = pyo.Set(initialize=list({country for date in D_Bids for t in D_Bids[date] for country in D_Bids[date][t]}))
model.D = pyo.RangeSet(1,dd)
model.L = pyo.Set(initialize=['Spain-Portugal'])

# Parameters
def kappa_init(m, l, z):
    start_zone, end_zone = l.split('-')
    return -1 if z==start_zone else 1 if z==end_zone else 0
def gas_index_init(m, d, b, t):
    z = next((zone for (zone, bb) in zone_mapping if bb == b), None)
    if z == 'Spain':
        block_numbers = bid_block_numbers1_ES
    elif z == 'Portugal':
        block_numbers = [block_number + N_Spain for block_number in bid_block_numbers1_PT]
    if (b in block_numbers and pyo.value(m.lambda_[d,b,t]) != 0):
        return gas_price.iloc[d-1]['Normalized_min'] ** 2
    return 0
model.Q = pyo.Param(model.D, model.B, model.T, initialize=lambda m, d, b, t: Q_final.get((b,d,t), 0), default=0)
model.lambda_ = pyo.Param(model.D, model.B, model.T, initialize=0, mutable=True)
model.F_max = pyo.Param(model.L, model.D, model.T, initialize=lambda m, l, d, t: all_border_exp[24*(d-1)+t-1]+1e-6)
model.F_min = pyo.Param(model.L, model.D, model.T, initialize=lambda m, l, d, t: -(all_border_exp[24*(d-1)+t-1]+1e-6))
model.R = pyo.Param(model.L, initialize=5000)
model.N_max = pyo.Param(model.Z, initialize=5000)
model.N_min = pyo.Param(model.Z, initialize=-5000)
model.Kappa = pyo.Param(model.L, model.Z, initialize=kappa_init)
if Elec_Gas_Coupling:
    model.gas_index = pyo.Param(model.D, model.B, model.T, initialize=gas_index_init)
else:
    model.gas_index = pyo.Param(model.D, model.B, model.T, initialize=0)

# Variables
model.x = pyo.Var(model.D, model.B, model.T)
model.f = pyo.Var(model.D, model.L, model.T)
model.n = pyo.Var(model.Z, model.D, model.T)

# Objective Function
model.con_a1 = pyo.Objective(
    expr=quicksum((model.lambda_[d, b, t] + model.gas_index[d, b, t]) * model.Q[d, b, t] * model.x[d, b, t] for d in model.D for b in model.B for t in model.T), sense=pyo.maximize)

# Constraints
def con_a4(m, z, d, t):
    return m.n[z,d,t] == quicksum(m.Q[d,b,t] * m.x[d,b,t] for (zb,b) in zone_mapping if zb==z)
model.con_a4 = pyo.Constraint(model.Z, model.D, model.T, rule=con_a4)
def con_a5(m, z, d, t):
    return m.n[z,d,t] == quicksum(m.Kappa[l,z] * m.f[d,l,t] for l in m.L)
model.con_a5 = pyo.Constraint(model.Z, model.D, model.T, rule=con_a5)
def con_a6(m, l, d, t):
    return (m.F_min[l,d,t], m.f[d,l,t], m.F_max[l,d,t])
model.con_a6 = pyo.Constraint(model.L, model.D, model.T, rule=con_a6)
def con_a7(m, l, d, t):
    if t == m.T.first():
        return pyo.Constraint.Skip
    return (-m.R[l], m.f[d,l,t] - m.f[d,l,t-1], m.R[l])
model.con_a7 = pyo.Constraint(model.L, model.D, model.T, rule=con_a7)
def con_a8(m, z, d, t):
    return (m.N_min[z], m.n[z,d,t], m.N_max[z])
model.con_a8 = pyo.Constraint(model.Z, model.D, model.T, rule=con_a8)
def con_a9(m, d, b, t):
    return (0, m.x[d,b,t], 1)
model.con_a9 = pyo.Constraint(model.D, model.B, model.T, rule=con_a9)
def lower_rule(m, d, b, t):
    return m.x[d, b, t] >= (1 - 0.0001) * X_final[b, d, t]
def upper_rule(m, d, b, t):
    return m.x[d, b, t] <= (1 + 0.0001) * X_final[b, d, t]
def lower_rule_MGA(m, d, b, t):
    if Q_final[b, d, t] > 0:
        return m.x[d, b, t] >= X_final[b, d, t]
    else:
        return pyo.Constraint.Skip
def upper_rule_MGA(m, d, b, t):
    if Q_final[b, d, t] > 0:
        return m.x[d, b, t] <= X_final[b, d, t]
    else:
        return pyo.Constraint.Skip
if not MGA:
    model.lower_rule = pyo.Constraint(model.D, model.B, model.T, rule=lower_rule)
    model.upper_rule = pyo.Constraint(model.D, model.B, model.T, rule=upper_rule)
else:
    model.lower_rule = pyo.Constraint(model.D, model.B, model.T, rule=lower_rule_MGA)
    model.upper_rule = pyo.Constraint(model.D, model.B, model.T, rule=upper_rule_MGA)

# Make the Model Environment
solver_FM = pyo.SolverFactory('gurobi')
num_continuous = sum(1 for v in model.component_data_objects(pyo.Var) if v.is_continuous())
num_binary = sum(1 for v in model.component_data_objects(pyo.Var) if v.is_binary())
num_constraints = sum(1 for c in model.component_data_objects(pyo.Constraint, active=True))
print(f"Number of continuous variables: {num_continuous}")
print(f"Number of binary variables: {num_binary}")
print(f"Number of constraints: {num_constraints}")
print("\nExecution time:", time.time() - start_time, "seconds")

In [ ]:
# Solve Forward Model
dual_values = {}
for file_idx, filename in enumerate(results_list):
    start_time = time.time()
    for z in model.Z:
        for d in model.D:
            for t in model.T:
                if z == 'Spain':
                    b_indices = range(0, int(N_Spain))
                elif z == 'Portugal':
                    b_indices = range(int(N_Spain), len(model.B))
                else:
                    b_indices = []
                for b_idx in b_indices:
                    b = list(model.B)[b_idx]
                    if model.Q[d,b,t] < 0 :
                        model.lambda_[d, b, t]= lambda_IO[(file_idx,z,b)]
                    elif model.Q[d,b,t] > 0:
                        if not MGA:
                            if z == 'Spain':
                                model.lambda_[d, b, t]= lambda_demand[(file_idx,z,d,t)]
                            elif z == 'Portugal':
                                model.lambda_[d, b, t]= lambda_demand[(file_idx,z,d,t)]
                        else:
                            model.lambda_[d, b, t]= 0
    results = solver_FM.solve(model, tee=False)
    if results.solver.status == pyo.SolverStatus.ok and results.solver.termination_condition == pyo.TerminationCondition.optimal:
        print(f"Optimal solution {file_idx+1} found : {pyo.value(model.con_a1)}")
        dual_values[file_idx] = {}
        for d in model.D:
            for t in model.T:
                dual_val_ES = model.dual[model.con_a5['Spain', d, t]]
                dual_val_PT = model.dual[model.con_a5['Portugal', d, t]]
                dual_values[file_idx][(d, t)] = (dual_val_ES, dual_val_PT)
    else:
        print(f"Solver did not find an optimal solution {file_idx+1}")
    print("Execution time:", time.time() - start_time, "seconds\n")

In [ ]:
# Comparison of Market Clearing Prices
plt.rcParams.update({
    'axes.titlesize': 20,
    'axes.labelsize': 20,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
    'legend.fontsize': 20,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
})
colors = [
    "#000000",
    "#984ea3",
    "#dede00",
    "#4daf4a",
    "#ff7f00",
    "#FF0000",
    "#999999",
    "#f781bf",
    "#0000FF",
    "#FEBF4B"
]
if MGA and Out_of_Smaple:
    specific_days_to_plot = [26, 42]  
    days = [d for d in model.D if d in specific_days_to_plot]
    hours = list(model.T)
    fig = plt.figure(figsize=(7.5*4, 6*30))
    plt.suptitle('Spain Zone', fontsize=50, y=0.993)  
    handles = []
    labels = []
    for idx_day, day in enumerate(days):
        ax = plt.subplot(30, 2, idx_day + 1)
        observed_mcp = [p for p in observed_mcp_ES[(day-1)*24:(day)*24]]
        line_actual, = ax.plot(
            hours,
            observed_mcp,
            label="Observed Market Price",
            color='blue',
            linestyle=':',
            linewidth=5,
            alpha=0.7
        )
        if idx_day == 0:
            handles.append(line_actual)
            labels.append("Observed Market Price")
        for file_idx in range(len(results_list)):
            filename=os.path.splitext(os.path.basename(results_list[file_idx]))[0]
            filename = filename.split('(')[0].rstrip()
            duals_day = []
            for hour in hours:
                dual_val = None
                if file_idx in dual_values and (day, hour) in dual_values[file_idx]:
                    dual_val = dual_values[file_idx][(day, hour)][0]
                duals_day.append(dual_val)
            line, = ax.plot(
                hours,
                duals_day,
                label=f"Produced Market Price via {filename}",
                color=colors[file_idx % len(colors)],
                linestyle='-',
                linewidth=5,
                alpha=0.7,
            )
            if idx_day == 0:
                handles.append(line)
                labels.append(f"Produced Market Price via {filename}")
        ax.set_title(f"Day {day}", fontsize=20)
        ax.set_xlabel("Time (h)", fontsize=20)
        ax.set_ylabel("Market-Clearing Price (€/MWh)", fontsize=20)
        ax.grid(True)
        ax.set_xticks(hours)
        ax.tick_params(axis='both', labelsize=15)
        ax.set_ylim(bottom=0)
    plt.tight_layout(rect=(0,0,1,0.95)) 
    fig.legend(handles, labels, loc='upper center', ncol=len(labels), fontsize=16, bbox_to_anchor=(0.5, 0.991), frameon=True)
    plt.subplots_adjust(top=0.9859)  
    plt.show()
elif MGA and not Out_of_Smaple:
    specific_days_to_plot = [18, 38]  
    days = [d for d in model.D if d in specific_days_to_plot]
    hours = list(model.T)
    fig = plt.figure(figsize=(7.5*4, 6*30))
    plt.suptitle('Spain Zone', fontsize=50, y=0.995)  
    handles = []
    labels = []
    for idx_day, day in enumerate(days):
        ax = plt.subplot(30, 2, idx_day + 1)
        observed_mcp = [p for p in observed_mcp_ES[(day-1)*24:(day)*24]]
        line_actual, = ax.plot(
            hours,
            observed_mcp,
            label="Observed Market Price",
            color='blue',
            linestyle=':',
            linewidth=5,
            alpha=0.7
        )
        if idx_day == 0:
            handles.append(line_actual)
            labels.append("Observed Market Price")
        for file_idx in range(len(results_list)):
            filename=os.path.splitext(os.path.basename(results_list[file_idx]))[0]
            filename = filename.split('(')[0].rstrip()
            duals_day = []
            for hour in hours:
                dual_val = None
                if file_idx in dual_values and (day, hour) in dual_values[file_idx]:
                    dual_val = dual_values[file_idx][(day, hour)][0]
                duals_day.append(dual_val)
            line, = ax.plot(
                hours,
                duals_day,
                label=f"Produced Market Price via {filename}",
                color=colors[file_idx % len(colors)],
                linestyle='-',
                linewidth=5,
                alpha=0.7,
            )
            if idx_day == 0:
                handles.append(line)
                labels.append(f"Produced Market Price via {filename}")
        ax.set_title(f"Day {day}", fontsize=20)
        ax.set_xlabel("Time (h)", fontsize=20)
        ax.set_ylabel("Market-Clearing Price (€/MWh)", fontsize=20)
        ax.grid(True)
        ax.set_xticks(hours)
        ax.tick_params(axis='both', labelsize=15)
        ax.set_ylim(bottom=0)
    plt.tight_layout(rect=(0,0,1,0.95)) 
    fig.legend(handles, labels, loc='upper center', ncol=len(labels), fontsize=16, bbox_to_anchor=(0.5, 0.991), frameon=True)
    plt.subplots_adjust(top=0.9859)  
    plt.show()
else:
    days = list(model.D)[:366]
    hours = list(model.T)
    fig = plt.figure(figsize=(7.5*4, 3*92))
    plt.suptitle('Spain Zone', fontsize=50, y=0.993)  
    handles = []
    labels = []
    for idx_day, day in enumerate(days):
        ax = plt.subplot(92, 4, idx_day + 1)
        observed_mcp = [p for p in observed_mcp_ES[idx_day*24:(idx_day+1)*24]]
        line_actual, = ax.plot(
            hours,
            observed_mcp,
            label="Observed Market Price",
            color='blue',
            linestyle=':',
            linewidth=5,
            alpha=0.7
        )
        if idx_day == 0:
            handles.append(line_actual)
            labels.append("Observed Market Price")
        for file_idx in range(len(results_list)):
            filename=os.path.splitext(os.path.basename(results_list[file_idx]))[0]
            filename = filename.split('(')[0].rstrip()
            duals_day = []
            for hour in hours:
                dual_val = None
                if file_idx in dual_values and (day, hour) in dual_values[file_idx]:
                    dual_val = dual_values[file_idx][(day, hour)][0]
                duals_day.append(dual_val)
            line, = ax.plot(
                hours,
                duals_day,
                label=f"Produced Market Price via {filename}",
                color=colors[file_idx % len(colors)],
                linestyle='-',
                linewidth=5,
                alpha=0.7,
            )
            if idx_day == 0:
                handles.append(line)
                labels.append(f"Produced Market Price via {filename}")
        ax.set_title(f"Day {day}", fontsize=20)
        ax.set_xlabel("Time (h)", fontsize=20)
        ax.set_ylabel("Market-Clearing Price (€/MWh)", fontsize=20)
        ax.grid(True)
        ax.set_xticks(hours)
        ax.tick_params(axis='both', labelsize=15)
        ax.set_ylim(bottom=0)
    plt.tight_layout(rect=(0,0,1,0.95)) 
    fig.legend(handles, labels, loc='upper center', ncol=len(labels), fontsize=16, bbox_to_anchor=(0.5, 0.99), frameon=True)
    plt.subplots_adjust(top=0.9859)  
    plt.show()

In [ ]:
# Comparison of Market Clearing Prices
if not MGA:
    fig = plt.figure(figsize=(7.5*4, 3*92))
    plt.suptitle('Portugal Zone', fontsize=50, y=0.993)
    handles = []
    labels = []
    for idx_day, day in enumerate(days):
        ax = plt.subplot(92, 4, idx_day + 1)
        observed_mcp = [p for p in observed_mcp_ES[idx_day*24:(idx_day+1)*24]]
        line_actual, = ax.plot(
            hours,
            observed_mcp,
            label="Observed Market Price",
            color='blue',
            linestyle=':',
            linewidth=5,
            alpha=0.7
        )
        if idx_day == 0:
            handles.append(line_actual)
            labels.append("Observed Market Price")
        for file_idx in range(len(results_list)):
            filename=os.path.splitext(os.path.basename(results_list[file_idx]))[0]
            filename = filename.split('(')[0].rstrip()
            duals_day = []
            for hour in hours:
                dual_val = None
                if file_idx in dual_values and (day, hour) in dual_values[file_idx]:
                    dual_val = dual_values[file_idx][(day, hour)][0]
                duals_day.append(dual_val)
            line, = ax.plot(
                hours,
                duals_day,
                label=f"Produced Market Price via {filename}",
                color=colors[file_idx % len(colors)],
                linestyle='-',
                linewidth=5,
                alpha=0.7,
            )
            if idx_day == 0:
                handles.append(line)
                labels.append(f"Produced Market Price via {filename}")
        ax.set_title(f"Day {day}", fontsize=20)
        ax.set_xlabel("Time (h)", fontsize=20)
        ax.set_ylabel("Market-Clearing Price (€/MWh)", fontsize=20)
        ax.grid(True)
        ax.set_xticks(hours)
        ax.tick_params(axis='both', labelsize=15)
        ax.set_ylim(bottom=0)
    plt.tight_layout(rect=(0,0,1,0.95)) 
    fig.legend(handles, labels, loc='upper center', ncol=len(labels), fontsize=16, bbox_to_anchor=(0.5, 0.99), frameon=True)
    plt.subplots_adjust(top=0.9859)  
    plt.show()

In [ ]:
# Calculate Error Indices
print('Spain Zone:')
observed_prices_ES = []
for idx_day, day in enumerate(model.D):
    observed_prices = [p for p in observed_mcp_ES[idx_day*24:(idx_day+1)*24]]
    for hour in hours:
        observed_prices_ES.append(observed_prices[hour-1])
duals_results_ES = {}
abs_results_ES = {}
mse_results_ES = {}
max_error_results_ES = {}
for file_idx in range(len(results_list)):
    filename=os.path.splitext(os.path.basename(results_list[file_idx]))[0]
    filename = filename.split('(')[0].rstrip()
    key_name = f"Produced Market Price via {filename}"
    duals_flat = []
    for idx_day, day in enumerate(model.D):
        for hour in hours:
            dual_val = dual_values[file_idx][(day, hour)][0] if (file_idx in dual_values and (day, hour) in dual_values[file_idx]) else None
            if dual_val is not None:
                duals_flat.append(dual_val)
    name_vs_actual = f"{key_name} vs Observed Market Price"
    duals_results_ES[name_vs_actual] = duals_flat
    errors_actual = np.abs(np.array(observed_prices_ES[:len(duals_flat)]) - np.array(duals_flat)) if duals_flat else np.array([])
    abs_results_ES[name_vs_actual] = np.sum(errors_actual) if duals_flat else np.nan
    mse_results_ES[name_vs_actual] = np.mean(errors_actual**2) if duals_flat else np.nan
    max_error_results_ES[name_vs_actual] = np.max(errors_actual) if errors_actual.size > 0 else np.nan
for name in mse_results_ES:
    if name.endswith("vs Observed Market Price"):
        max_value = max_error_results_ES.get(name, np.nan)
        print(f"Error between {name}-> Cumulative Error: {round(abs_results_ES[name], 2)} -- MSE: {round(mse_results_ES[name], 2)}")


print('\nPortugal Zone:')
observed_prices_PT = []
for idx_day, day in enumerate(model.D):
    observed_prices = [p for p in observed_mcp_PT[idx_day*24:(idx_day+1)*24]]
    for hour in hours:
        observed_prices_PT.append(observed_prices[hour-1])
duals_results_PT = {}
abs_results_PT = {}
mse_results_PT = {}
max_error_results_PT = {}
for file_idx in range(len(results_list)):
    filename=os.path.splitext(os.path.basename(results_list[file_idx]))[0]
    filename = filename.split('(')[0].rstrip()
    key_name = f"Produced Market Price via {filename}"
    duals_flat = []
    for idx_day, day in enumerate(model.D):
        for hour in hours:
            dual_val = dual_values[file_idx][(day, hour)][1] if (file_idx in dual_values and (day, hour) in dual_values[file_idx]) else None
            if dual_val is not None:
                duals_flat.append(dual_val)
    name_vs_actual = f"{key_name} vs Observed Market Price"
    duals_results_PT[name_vs_actual] = duals_flat
    errors_actual = np.abs(np.array(observed_prices_PT[:len(duals_flat)]) - np.array(duals_flat)) if duals_flat else np.array([])
    abs_results_PT[name_vs_actual] = np.sum(errors_actual) if duals_flat else np.nan
    mse_results_PT[name_vs_actual] = np.mean(errors_actual**2) if duals_flat else np.nan
    max_error_results_PT[name_vs_actual] = np.max(errors_actual) if errors_actual.size > 0 else np.nan
for name in mse_results_PT:
    if name.endswith("vs Observed Market Price"):
        max_value = max_error_results_PT.get(name, np.nan)
        print(f"Error between {name}-> Cumulative Error: {round(abs_results_PT[name], 2)} -- MSE: {round(mse_results_PT[name], 2)}")